In [147]:
!pip install librosa

In [148]:
import librosa
import soundfile
import os
import shutil
import numpy as np

def create_base_dataset(source_path: str, destination_path: str):
    sources = os.listdir(source_path)
    if os.path.exists(destination_path):
        shutil.rmtree(destination_path)
    os.mkdir(destination_path)

    print("Found files:")
    print(sources)

    for i, filename in enumerate(sources):
        path = os.path.join(source_path, filename)
        file, sampling_rate = librosa.load(path)

        # 2. Track short-term energy using Root Mean Square (RMS)
        # Breaths show up as low-energy blocks between high-energy spoken words
        rms = librosa.feature.rms(y=file, frame_length=2048, hop_length=512)[0]

        # Convert RMS to Decibels
        rms_db = librosa.amplitude_to_db(rms, ref=np.max)

        # 3. Create a gain mask
        # Thresholds depend on recording quality. Breaths usually sit between -30dB and -45dB.
        # Anything below -45dB is room noise. Anything above -25dB is active speech.
        breath_threshold_upper = -15.0
        breath_threshold_lower = -40.0

        # Initialize an array of 1.0 (no volume change) mapped to the audio length
        gain_mask = np.ones_like(file)

        # Loop through frames to locate breath regions
        for frame_idx, db_level in enumerate(rms_db):
            if breath_threshold_lower < db_level < breath_threshold_upper:
                # Determine the sample boundaries for this frame
                start_sample = frame_idx * 512
                end_sample = start_sample + 2048

                # Attenuate the volume in this frame by roughly 15dB (multiply by ~0.17)
                gain_mask[start_sample:end_sample] = 0.4

        # 4. Apply the filter and export
        file_filtered = file * gain_mask

        intervals = librosa.effects.split(file_filtered, frame_length=librosa.time_to_frames(5, sr=sampling_rate), top_db=50)

        filtered_intervals = [interval for interval in intervals if interval[1] - interval[0] > 8000]
        print(filtered_intervals)

        for j, interval in enumerate(filtered_intervals):
            a, b = interval
            sample = file[a:b]

            soundfile.write(f"{destination_path}/{i}_{j}.wav", sample, samplerate=int(sampling_rate))


create_base_dataset("./sonya_source", './sonya_base')

Found files:
['Sonya_1.wav', 'Sonya_2.wav']
[array([20992, 63488]), array([65024, 79872]), array([ 80384, 183808]), array([185344, 213504]), array([214528, 225280]), array([226304, 280576]), array([281088, 365568]), array([366080, 397312]), array([397824, 429056]), array([430080, 441856]), array([442368, 467968]), array([468992, 492032]), array([492544, 502272]), array([502784, 558080]), array([559104, 601088]), array([604672, 621568]), array([623104, 655872]), array([656896, 672768]), array([673792, 754688]), array([755712, 784896]), array([788480, 818176]), array([819712, 841728]), array([843776, 920064])]
[array([  2048, 704448])]
